In [1]:
# ========== 0) 드라이브 마운트 ==========
from google.colab import drive
drive.mount('/content/drive')

# ========== 1) 경로/임포트 ==========
from pathlib import Path
import os, re, zipfile, pandas as pd

ROOT_DIR    = Path('/content/drive/MyDrive/IMU_DATA_K')   # 드라이브 내 작업 폴더
EXTRACT_DIR = Path('/content/IMU_DATA_extracted')       # Colab 런타임 로컬
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# ========== 2) ZIP 자동 탐색 & 해제 (없으면 스킵) ==========
zip_candidates = sorted(ROOT_DIR.glob('*.zip'), key=lambda p: p.stat().st_mtime, reverse=True)
if zip_candidates:
    ZIP_PATH = zip_candidates[0]
    print(f"[INFO] 사용할 ZIP: {ZIP_PATH.name}")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[OK] 압축 해제 완료 → {EXTRACT_DIR.resolve()}")
else:
    print(f"[WARN] ZIP을 찾지 못했습니다: {ROOT_DIR}  (이미 해제된 폴더를 사용합니다)")

# ========== 3) label_* 폴더의 '공통 상위 경로' 자동 탐색 ==========
def _is_label_dir(p: Path):
    name = p.name.lower()
    # label_*, class_*, good/bad 형태 지원
    return (
        name.startswith('label_') or
        name.startswith('class_') or
        name in {'good','bad','positive','negative'}
    )

def _list_label_dirs(base: Path):
    # rglob로 모든 하위 폴더 탐색
    return [d for d in base.rglob('*') if d.is_dir() and _is_label_dir(d)]

def _common_parent(paths):
    # 여러 경로의 공통 상위 경로 반환
    if not paths: return None
    common = os.path.commonpath([str(p) for p in paths])
    return Path(common)

def find_dataset_root(extracted: Path, fallback_root: Path):
    # 1) 해제 경로에서 label 디렉토리 찾기
    label_dirs = _list_label_dirs(extracted)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto): {parent}")
        return parent
    # 2) 드라이브 루트 하위에서 직접 찾기(이미 해제된 경우)
    label_dirs = _list_label_dirs(fallback_root)
    if label_dirs:
        parent = _common_parent([d.parent for d in label_dirs])
        print(f"[INFO] DATASET_ROOT(auto from DRIVE): {parent}")
        return parent
    raise FileNotFoundError("[ERROR] 'label_*' 또는 유사 폴더를 찾지 못했습니다.")

DATASET_ROOT = find_dataset_root(EXTRACT_DIR, ROOT_DIR)

# ========== 4) 라벨 매핑 자동화 + 안전 CSV 로딩 ==========
def _infer_label_map(root: Path):
    """root 하위의 label 디렉토리를 자동 매핑: label_0→0, label_1→1, ... / 그 외는 사전식 정렬 순서대로 0..K-1"""
    subdirs = [d for d in sorted(root.iterdir()) if d.is_dir() and _is_label_dir(d)]
    if not subdirs:
        # 한 단계 더 내려가서 재탐색
        subdirs = sorted([d for d in root.rglob('*') if d.is_dir() and _is_label_dir(d)], key=lambda p: p.as_posix())
    if not subdirs:
        raise FileNotFoundError("[ERROR] 라벨 디렉토리를 찾지 못했습니다.")
    mapping = {}
    for d in subdirs:
        name = d.name
        m = re.match(r'^(?:label|class)_(\d+)$', name, flags=re.IGNORECASE)
        if m:
            y = int(m.group(1))
        else:
            # good/bad 등은 사전식 정렬 순으로 0..K-1
            # 단, 관례상 'bad/negative'를 0, 'good/positive'를 1로 맞추려면 아래 우선순위 가중치 제공
            order_bias = {'bad':0, 'negative':0, 'good':1, 'positive':1}
            y = order_bias.get(name.lower(), None)
            if y is None:
                # 임시 None이면 나중에 정렬해서 인덱스 부여
                y = None
        mapping[name] = y
    # None이 남아있으면 사전식 정렬 순서대로 빈 ID를 채움
    used = {v for v in mapping.values() if v is not None}
    next_ids = [i for i in range(len(mapping)) if i not in used]
    for k in sorted([k for k,v in mapping.items() if v is None]):
        mapping[k] = next_ids.pop(0)
    return mapping

def _read_csv_safely(path: Path, encoding_pref=('utf-8-sig','cp949','utf-8')):
    last_err = None
    for enc in encoding_pref:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    # 마지막 시도로 인코딩 미지정
    try:
        return pd.read_csv(path)
    except Exception:
        raise last_err

def load_labeled_imu(root_dir: Path,
                     label_map='auto',
                     pattern=('*.csv', '*.CSV'),
                     verbose=True,
                     preview_files=5):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"[ERROR] 루트가 없습니다: {root}")

    if label_map == 'auto':
        label_map_used = _infer_label_map(root)
    else:
        label_map_used = dict(label_map)

    print(f"[INFO] 라벨 매핑: {label_map_used}")

    total_files, dfs = 0, []
    for subdir_name, y in label_map_used.items():
        d = root / subdir_name
        if not (d.exists() and d.is_dir()):
            print(f"[MISS] 폴더 없음: {d}")
            continue
        # 여러 패턴 지원
        files = []
        for pat in pattern:
            files += list(d.glob(pat))
        files = sorted(set(files))
        n = len(files); total_files += n
        print(f"[OK] {d.name} → {n}개 파일")
        if verbose and n>0:
            for f in files[:preview_files]:
                print(f"    • {f.name}")
            if n > preview_files:
                print(f"    • ... (총 {n}개)")
        for f in files:
            try:
                df = _read_csv_safely(f)
            except Exception as e:
                print(f"[WARN] 읽기 실패: {f.name} ({e})"); continue
            # 메타정보
            df['label']       = y
            df['source_file'] = f.name
            df['source_dir']  = subdir_name
            dfs.append(df)

    if total_files == 0 or not dfs:
        raise FileNotFoundError("[ERROR] CSV를 찾지 못했습니다. 경로/패턴을 확인하세요.")

    data = pd.concat(dfs, axis=0, ignore_index=True, sort=False)

    # 로딩 품질 점검 로그(선택)
    def _normalize_name(s):
        s = re.sub(r'[^A-Za-z0-9]+',' ', str(s)).strip().lower()
        return re.sub(r'\s+',' ', s)
    cols_norm = {_normalize_name(c): c for c in data.columns}
    move_candidates = [c for c in data.columns if _normalize_name(c) in ['ak','move','rep','segment','cycle','trial','action']]
    acc_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['acc','accelerometer'])]
    gyr_candidates  = [c for c in data.columns if any(k in _normalize_name(c) for k in ['gyr','gyro'])]

    print("\n[SUMMARY]")
    print(f" - 총 CSV 파일 수: {total_files}개")
    print(f" - 총 로우 수: {len(data):,}")
    print(f" - 라벨 분포:\n{data['label'].value_counts(dropna=False).to_string()}")
    if move_candidates:
        print(f" - move 관련 열 후보: {move_candidates}")
    else:
        print(" - [WARN] move(AK) 열을 찾지 못했습니다. 전처리 전에 열 이름을 확인하세요.")
    print(f" - 가속도 열 후보: {acc_candidates[:6]}")
    print(f" - 자이로 열 후보: {gyr_candidates[:6]}")

    return data, label_map_used

# ========== 5) 실제 로딩 ==========
imu_df, label_map_used = load_labeled_imu(
    DATASET_ROOT,
    label_map='auto',            # 필요하면 {'label_0':0,'label_1':1}로 고정 가능
    pattern=('*.csv','*.CSV'),
    verbose=True,
    preview_files=5
)

# ========== 6) (옵션) 업로드 CSV 폴백 병합 ==========
# Colab 외부에서 개별 CSV를 올려둔 경우(예: ChatGPT에서 제공) 병합
fallback_csvs = [Path('/mnt/data/IMU_Label_Plus_ALL.csv'), Path('/mnt/data/IMU_label_1_jungro.csv')]
fallback_exist = [p for p in fallback_csvs if p.exists()]
if fallback_exist:
    add_dfs = []
    for p in fallback_exist:_


Mounted at /content/drive
[INFO] 사용할 ZIP: ekf_appended_csv_only (2).zip
[OK] 압축 해제 완료 → /content/IMU_DATA_extracted
[INFO] DATASET_ROOT(auto): /content/IMU_DATA_extracted
[INFO] 라벨 매핑: {'label_0': 0, 'label_1': 1}
[OK] label_0 → 1개 파일
    • IMU_Label_Plus_ALL_ekf.csv
[OK] label_1 → 1개 파일
    • IMU_label_1_jungro_ekf.csv

[SUMMARY]
 - 총 CSV 파일 수: 2개
 - 총 로우 수: 45,143
 - 라벨 분포:
label
0    36535
1     8608
 - move 관련 열 후보: ['move']
 - 가속도 열 후보: ['acc_lin_x', 'acc_lin_y', 'acc_lin_z']
 - 자이로 열 후보: []


In [5]:
# ===================== TCN Training (Cube.AI-friendly, 80/10/10 MOVE split) =====================
# - Compatible with STM32Cube.AI
# - Re-splits dataset per preset into 80%/10%/10% (train/val/test) at MOVE level
# - Saves .keras and .h5 (float32) models, plus F1 scores & reports
# ===============================================================================================

import os, json, math, time, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedShuffleSplit

# -------------------- Config --------------------
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# Dataset location (use the NPZ files built from your EKF→MOVE pipeline)
BASE_DIR   = Path('/content/IMU_DATA_extracted/npz_from_ekf')
L          = 128
PRESETS    = ['6ch', '9ch', '12ch']              # train if the NPZ exists
NPZ_NAME   = lambda preset: BASE_DIR / f"ekf_move_L{L}_{preset}.npz"

# Split ratios (MOVE-level)
VAL_RATIO  = 0.10
TEST_RATIO = 0.10

# Training
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 3e-3
PATIENCE   = 8
MIN_VALID_RATIO = 0.60        # drop sequences with too much pad (mask mean < 0.6)

# Model capacity (keep small for MCU)
WIDTH        = 48
KERNEL_SIZE  = 5
DILATIONS    = [1, 2, 4, 8]
USE_BN       = True
DROPOUT_RATE = 0.05
CAUSAL       = False

OUT_DIR = Path('/content/tcn_runs'); OUT_DIR.mkdir(parents=True, exist_ok=True)

# -------------------- Utils --------------------
def load_all_from_npz(npz_path: Path):
    """Load all splits from NPZ and merge to a single (X,y,M) — each row is one MOVE."""
    d = np.load(npz_path, allow_pickle=True)
    X = np.concatenate([d['X_train'], d['X_val'], d['X_test']], axis=0)
    y = np.concatenate([d['y_train'], d['y_val'], d['y_test']], axis=0)
    M = np.concatenate([d['mask_train'], d['mask_val'], d['mask_test']], axis=0)
    return X, y, M

def filter_by_valid_ratio(X, y, M, thr=MIN_VALID_RATIO):
    """Keep samples whose valid mask mean >= thr."""
    if M is None:
        return X, y, M
    vr = M.mean(axis=1)  # (N,)
    keep = vr >= thr
    return X[keep], y[keep], (M[keep] if M is not None else None)

def stratified_split_indices(y, val_ratio=0.1, test_ratio=0.1, seed=SEED):
    """Return train/val/test indices with stratification (80/10/10 by default)."""
    n = len(y)
    assert 0 < test_ratio < 1 and 0 < val_ratio < 1 and test_ratio + val_ratio < 1
    # 1) Take TEST from all
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=seed)
    train_idx, test_idx = next(sss_test.split(np.zeros(n), y))
    # 2) From remaining, take VAL with relative ratio
    y_rem = y[train_idx]
    rel_val = val_ratio / (1.0 - test_ratio)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=rel_val, random_state=seed)
    tr_idx_rel, va_idx_rel = next(sss_val.split(np.zeros(len(y_rem)), y_rem))
    val_idx   = train_idx[va_idx_rel]
    train_idx = train_idx[tr_idx_rel]
    return train_idx, val_idx, test_idx

def make_class_weights(y):
    classes = np.unique(y)
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, cw)}

def save_and_show_reports(y_true, y_pred, labels, out_txt):
    """Save to file + print to console, including F1 macro/weighted."""
    rep = classification_report(y_true, y_pred, digits=3)
    cm  = confusion_matrix(y_true, y_pred, labels=labels)
    f1m = f1_score(y_true, y_pred, average='macro')
    f1w = f1_score(y_true, y_pred, average='weighted')

    with open(out_txt, 'w', encoding='utf-8') as f:
        f.write(rep + "\n")
        f.write("Confusion Matrix (rows=true, cols=pred):\n")
        f.write(np.array2string(cm, separator=' ') + "\n")
        f.write(f"F1-macro: {f1m:.4f}, F1-weighted: {f1w:.4f}\n")

    print("\n" + "="*12 + f" Report @ {out_txt.name} " + "="*12)
    print(rep)
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print(f"[F1] macro={f1m:.4f}, weighted={f1w:.4f}")
    return {"f1_macro": float(f1m), "f1_weighted": float(f1w)}

# -------------------- TCN Model (on-device safe) --------------------
def NormOrIdentity():
    return layers.BatchNormalization() if USE_BN else layers.Activation('linear')

def tcn_block(x, filters, kernel_size, dilation_rate, name_prefix):
    pad_type = 'causal' if CAUSAL else 'same'
    h = layers.Conv1D(filters, kernel_size, padding=pad_type,
                      dilation_rate=dilation_rate, use_bias=not USE_BN,
                      name=f"{name_prefix}_conv1")(x)
    h = NormOrIdentity()(h)
    h = layers.Activation('relu')(h)
    if DROPOUT_RATE and DROPOUT_RATE > 0:
        h = layers.Dropout(DROPOUT_RATE)(h)

    h = layers.Conv1D(filters, kernel_size, padding=pad_type,
                      dilation_rate=dilation_rate, use_bias=not USE_BN,
                      name=f"{name_prefix}_conv2")(h)
    h = NormOrIdentity()(h)

    # Residual
    if x.shape[-1] != filters:
        res = layers.Conv1D(filters, 1, padding='same', use_bias=True,
                            name=f"{name_prefix}_res")(x)
    else:
        res = x

    h = layers.Add(name=f"{name_prefix}_add")([res, h])
    h = layers.Activation('relu')(h)
    return h

def build_tcn(input_length, n_channels, n_classes):
    inp = keras.Input(shape=(input_length, n_channels), name='x')
    x = inp
    x = layers.Conv1D(WIDTH, 3, padding='same', use_bias=not USE_BN, name='stem_conv')(x)
    x = NormOrIdentity()(x)
    x = layers.Activation('relu')(x)
    f = WIDTH
    for i, d in enumerate(DILATIONS):
        x = tcn_block(x, f, KERNEL_SIZE, d, name_prefix=f"tcn{i+1}")
        if i == len(DILATIONS)//2:
            f = int(f * 1.5)
    x = layers.GlobalAveragePooling1D(name='gap')(x)   # Cube.AI-friendly
    out = layers.Dense(n_classes, activation='softmax', name='pred')(x)
    return keras.Model(inp, out, name='tcn_cls')

# -------------------- Training Loop per preset --------------------
def train_one_preset(npz_path: Path, preset: str):
    if not npz_path.exists():
        print(f"[SKIP] {preset}: NPZ not found → {npz_path}")
        return

    print(f"\n========== Training {preset} (MOVE-level 80/10/10) ==========")
    # 0) Load & merge all moves
    X_all, y_all, M_all = load_all_from_npz(npz_path)

    # 1) Filter by valid ratio BEFORE split
    X_all, y_all, M_all = filter_by_valid_ratio(X_all, y_all, M_all, thr=MIN_VALID_RATIO)

    # 2) Stratified MOVE-level 80/10/10 split
    tr_idx, va_idx, te_idx = stratified_split_indices(y_all, VAL_RATIO, TEST_RATIO, seed=SEED)
    Xtr, ytr, Mtr = X_all[tr_idx], y_all[tr_idx], (M_all[tr_idx] if M_all is not None else None)
    Xva, yva, Mva = X_all[va_idx], y_all[va_idx], (M_all[va_idx] if M_all is not None else None)
    Xte, yte, Mte = X_all[te_idx], y_all[te_idx], (M_all[te_idx] if M_all is not None else None)

    n_classes  = int(np.max([ytr.max(), yva.max(), yte.max()])) + 1
    n_channels = Xtr.shape[-1]
    assert Xtr.shape[1] == L, f"L mismatch: expected {L}, got {Xtr.shape[1]}"

    # 3) Build & compile
    model = build_tcn(L, n_channels, n_classes)
    opt = keras.optimizers.Adam(learning_rate=LR)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # 4) Callbacks & dirs
    run_dir = OUT_DIR / f"{preset}_L{L}_{n_channels}ch"
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / "best.keras"
    cbs = [
        keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor='val_accuracy',
                                        save_best_only=True, mode='max', verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=max(2, PATIENCE//3), min_lr=1e-5, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                                      mode='max', restore_best_weights=True, verbose=1),
    ]

    # 5) Class weights (imbalanced)
    cw = make_class_weights(ytr)

    # 6) Train
    hist = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=cbs,
        verbose=2
    )

    # 7) Evaluate
    va_eval = model.evaluate(Xva, yva, verbose=0)
    te_eval = model.evaluate(Xte, yte, verbose=0)
    print(f"[VAL] loss={va_eval[0]:.4f}, acc={va_eval[1]:.4f}")
    print(f"[TEST] loss={te_eval[0]:.4f}, acc={te_eval[1]:.4f}")

    # 8) Reports (with F1)
    yv_pred = np.argmax(model.predict(Xva, batch_size=256, verbose=0), axis=1)
    yt_pred = np.argmax(model.predict(Xte, batch_size=256, verbose=0), axis=1)
    val_metrics  = save_and_show_reports(yva, yv_pred, labels=list(range(n_classes)),
                                         out_txt=run_dir / "val_report.txt")
    test_metrics = save_and_show_reports(yte, yt_pred, labels=list(range(n_classes)),
                                         out_txt=run_dir / "test_report.txt")

    # 9) Save final model (.keras + .h5 float32)
    final_keras = run_dir / "final.keras"
    model.save(final_keras, include_optimizer=False)

    model_f32 = keras.models.clone_model(model)
    model_f32.build(model.input_shape)
    model_f32.set_weights([w.astype(np.float32) for w in model.get_weights()])
    h5_path = run_dir / "final_export_cubeai.h5"
    model_f32.save(h5_path, include_optimizer=False)
    print("[SAVED]", final_keras)
    print("[SAVED]", h5_path)

    # 10) Save training log (include F1s)
    with open(run_dir / "train_summary.json", 'w') as f:
        json.dump({
            "preset": preset,
            "L": int(L),
            "channels": int(n_channels),
            "val": {
                "loss": float(va_eval[0]),
                "acc": float(va_eval[1]),
                "f1_macro": val_metrics["f1_macro"],
                "f1_weighted": val_metrics["f1_weighted"]
            },
            "test": {
                "loss": float(te_eval[0]),
                "acc": float(te_eval[1]),
                "f1_macro": test_metrics["f1_macro"],
                "f1_weighted": test_metrics["f1_weighted"]
            },
            "class_weight": {int(k): float(v) for k, v in cw.items()},
            "history": {k: [float(x) for x in v] for k, v in hist.history.items()}
        }, f, indent=2)

if __name__ == "__main__":
    for preset in PRESETS:
        try:
            npz_path = NPZ_NAME(preset)
            train_one_preset(npz_path, preset)
        except Exception as e:
            print(f"[ERROR] {preset}: {e}")



========== Training 6ch (MOVE-level 80/10/10) ==========
Epoch 1/50

Epoch 1: val_accuracy improved from -inf to 0.77419, saving model to /content/tcn_runs/6ch_L128_6ch/best.keras
20/20 - 22s - 1s/step - accuracy: 0.5674 - loss: 0.7361 - val_accuracy: 0.7742 - val_loss: 0.6234 - learning_rate: 0.0030
Epoch 2/50

Epoch 2: val_accuracy did not improve from 0.77419
20/20 - 0s - 20ms/step - accuracy: 0.6015 - loss: 0.6226 - val_accuracy: 0.5032 - val_loss: 0.6037 - learning_rate: 0.0030
Epoch 3/50

Epoch 3: val_accuracy did not improve from 0.77419
20/20 - 0s - 23ms/step - accuracy: 0.5455 - loss: 0.6221 - val_accuracy: 0.4903 - val_loss: 0.5930 - learning_rate: 0.0030
Epoch 4/50

Epoch 4: val_accuracy did not improve from 0.77419
20/20 - 0s - 14ms/step - accuracy: 0.6039 - loss: 0.6016 - val_accuracy: 0.4839 - val_loss: 0.6036 - learning_rate: 0.0030
Epoch 5/50

Epoch 5: val_accuracy did not improve from 0.77419

Epoch 5: ReduceLROnPlateau reducing learning rate to 0.001500000013038516.



============ Report @ val_report.txt ============
              precision    recall  f1-score   support

           0      0.831     0.904     0.866       125
           1      0.368     0.233     0.286        30

    accuracy                          0.774       155
   macro avg      0.600     0.569     0.576       155
weighted avg      0.741     0.774     0.754       155

Confusion Matrix (rows=true, cols=pred):
[[113  12]
 [ 23   7]]
[F1] macro=0.5758, weighted=0.7536

============ Report @ test_report.txt ============
              precision    recall  f1-score   support

           0      0.824     0.864     0.844       125
           1      0.292     0.233     0.259        30

    accuracy                          0.742       155
   macro avg      0.558     0.549     0.552       155
weighted avg      0.721     0.742     0.731       155

Confusion Matrix (rows=true, cols=pred):
[[108  17]
 [ 23   7]]
[F1] macro=0.5515, weighted=0.7306


[SAVED] /content/tcn_runs/6ch_L128_6ch/final.keras
[SAVED] /content/tcn_runs/6ch_L128_6ch/final_export_cubeai.h5
[SKIP] 9ch: NPZ not found → /content/IMU_DATA_extracted/npz_from_ekf/ekf_move_L128_9ch.npz
[SKIP] 12ch: NPZ not found → /content/IMU_DATA_extracted/npz_from_ekf/ekf_move_L128_12ch.npz


In [4]:
# ===================== TCN Training (Cube.AI-friendly) =====================
# - Compatible layers for on-device (STM32Cube.AI)
# - Trains 6ch / 9ch / 12ch presets separately (one NPZ per preset)
# - Saves .keras and .h5 (float32) models for export
# - Prints and saves F1 scores & classification reports
# ==========================================================================

import os, json, glob, math, time, random
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score  # <-- ADD

# -------------------- Config --------------------
SEED = 42
np.random.seed(SEED); random.seed(SEED); tf.random.set_seed(SEED)

# Dataset location (use the NPZ files you created from EKF pipeline)
BASE_DIR = Path('/content/IMU_DATA_extracted/npz_from_ekf')
L         = 128                               # must match your NPZ
PRESETS   = ['6ch', '9ch', '12ch']            # train per preset if file exists
NPZ_NAME  = lambda preset: BASE_DIR / f"ekf_move_L{L}_{preset}.npz"

# Training
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 3e-3
PATIENCE   = 8
MIN_VALID_RATIO = 0.60        # drop sequences with too much pad (mask mean < 0.6)

# Model capacity (keep it small for MCU)
WIDTH        = 48             # base channels (try 32~64)
KERNEL_SIZE  = 5              # 3 or 5 recommended
DILATIONS    = [1, 2, 4, 8]   # TCN dilation stack
USE_BN       = True           # BatchNorm (Cube.AI fuses; safe to keep)
DROPOUT_RATE = 0.05           # light dropout (train only)
CAUSAL       = False          # causal=False is fine for window classification

OUT_DIR = Path('/content/tcn_runs'); OUT_DIR.mkdir(parents=True, exist_ok=True)


# -------------------- Utils --------------------
def load_npz(npz_path: Path):
    d = np.load(npz_path, allow_pickle=True)
    Xtr, ytr, Mtr = d['X_train'], d['y_train'], d['mask_train']
    Xva, yva, Mva = d['X_val'],   d['y_val'],   d['mask_val']
    Xte, yte, Mte = d['X_test'],  d['y_test'],  d['mask_test']
    return (Xtr, ytr, Mtr), (Xva, yva, Mva), (Xte, yte, Mte)

def filter_by_valid_ratio(X, y, M, thr=MIN_VALID_RATIO):
    if M is None: return X, y, M
    vr = M.mean(axis=1)  # (N,)
    keep = vr >= thr
    return X[keep], y[keep], (M[keep] if M is not None else None)

def make_class_weights(y):
    classes = np.unique(y)
    cw = compute_class_weight(class_weight='balanced', classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, cw)}

def save_and_show_reports(y_true, y_pred, labels, out_txt):
    """파일로 저장 + 콘솔 출력 (F1 macro/weighted 포함)"""
    rep = classification_report(y_true, y_pred, digits=3)
    cm  = confusion_matrix(y_true, y_pred, labels=labels)
    f1m = f1_score(y_true, y_pred, average='macro')
    f1w = f1_score(y_true, y_pred, average='weighted')

    # 파일 저장
    with open(out_txt, 'w', encoding='utf-8') as f:
        f.write(rep + "\n")
        f.write("Confusion Matrix (rows=true, cols=pred):\n")
        f.write(np.array2string(cm, separator=' ') + "\n")
        f.write(f"F1-macro: {f1m:.4f}, F1-weighted: {f1w:.4f}\n")

    # 콘솔 출력
    print("\n" + "="*12 + f" Report @ {out_txt.name} " + "="*12)
    print(rep)
    print("Confusion Matrix (rows=true, cols=pred):")
    print(cm)
    print(f"[F1] macro={f1m:.4f}, weighted={f1w:.4f}")
    return {"f1_macro": float(f1m), "f1_weighted": float(f1w)}


# -------------------- TCN Model (on-device safe) --------------------
def NormOrIdentity():
    return layers.BatchNormalization() if USE_BN else layers.Activation('linear')

def tcn_block(x, filters, kernel_size, dilation_rate, name_prefix):
    # Conv -> BN -> ReLU -> Dropout -> Conv -> BN -> ReLU, with residual 1x1
    pad_type = 'causal' if CAUSAL else 'same'
    h = layers.Conv1D(filters, kernel_size, padding=pad_type,
                      dilation_rate=dilation_rate, use_bias=not USE_BN,
                      name=f"{name_prefix}_conv1")(x)
    h = NormOrIdentity()(h)
    h = layers.Activation('relu')(h)
    if DROPOUT_RATE and DROPOUT_RATE > 0:
        h = layers.Dropout(DROPOUT_RATE)(h)

    h = layers.Conv1D(filters, kernel_size, padding=pad_type,
                      dilation_rate=dilation_rate, use_bias=not USE_BN,
                      name=f"{name_prefix}_conv2")(h)
    h = NormOrIdentity()(h)

    # Residual (match channels if needed)
    if x.shape[-1] != filters:
        res = layers.Conv1D(filters, 1, padding='same', use_bias=True,
                            name=f"{name_prefix}_res")(x)
    else:
        res = x

    h = layers.Add(name=f"{name_prefix}_add")([res, h])
    h = layers.Activation('relu')(h)
    return h

def build_tcn(input_length, n_channels, n_classes):
    inp = keras.Input(shape=(input_length, n_channels), name='x')
    x = inp

    # Stem
    x = layers.Conv1D(WIDTH, 3, padding='same', use_bias=not USE_BN, name='stem_conv')(x)
    x = NormOrIdentity()(x)
    x = layers.Activation('relu')(x)

    # Dilation stack
    f = WIDTH
    for i, d in enumerate(DILATIONS):
        x = tcn_block(x, f, KERNEL_SIZE, d, name_prefix=f"tcn{i+1}")
        # Optional width growth (mild)
        if i == len(DILATIONS)//2:
            f = int(f * 1.5)

    # Head
    x = layers.GlobalAveragePooling1D(name='gap')(x)  # Cube.AI-friendly
    out = layers.Dense(n_classes, activation='softmax', name='pred')(x)

    model = keras.Model(inp, out, name='tcn_cls')
    return model


# -------------------- Training Loop per preset --------------------
def train_one_preset(npz_path: Path, preset: str):
    if not npz_path.exists():
        print(f"[SKIP] {preset}: NPZ not found → {npz_path}")
        return

    print(f"\n========== Training {preset} ==========")
    (Xtr, ytr, Mtr), (Xva, yva, Mva), (Xte, yte, Mte) = load_npz(npz_path)

    # Filter by valid ratio (drop heavy-padded)
    Xtr, ytr, Mtr = filter_by_valid_ratio(Xtr, ytr, Mtr, thr=MIN_VALID_RATIO)
    Xva, yva, Mva = filter_by_valid_ratio(Xva, yva, Mva, thr=MIN_VALID_RATIO)
    Xte, yte, Mte = filter_by_valid_ratio(Xte, yte, Mte, thr=MIN_VALID_RATIO)

    n_classes = int(np.max([ytr.max(), yva.max(), yte.max()])) + 1
    n_channels = Xtr.shape[-1]
    assert Xtr.shape[1] == L, f"L mismatch: expected {L}, got {Xtr.shape[1]}"

    # Build model
    model = build_tcn(L, n_channels, n_classes)
    opt = keras.optimizers.Adam(learning_rate=LR)
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

    # Callbacks
    run_dir = OUT_DIR / f"{preset}_L{L}_{n_channels}ch"
    run_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = run_dir / "best.keras"
    cbs = [
        keras.callbacks.ModelCheckpoint(str(ckpt_path), monitor='val_accuracy',
                                        save_best_only=True, mode='max', verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=max(2, PATIENCE//3), min_lr=1e-5, verbose=1),
        keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=PATIENCE,
                                      mode='max', restore_best_weights=True, verbose=1),
    ]

    # Class weights for imbalance
    cw = make_class_weights(ytr)

    # Train
    hist = model.fit(
        Xtr, ytr,
        validation_data=(Xva, yva),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw,
        callbacks=cbs,
        verbose=2
    )

    # Evaluate
    va_eval = model.evaluate(Xva, yva, verbose=0)
    te_eval = model.evaluate(Xte, yte, verbose=0)
    print(f"[VAL] loss={va_eval[0]:.4f}, acc={va_eval[1]:.4f}")
    print(f"[TEST] loss={te_eval[0]:.4f}, acc={te_eval[1]:.4f}")

    # Predictions
    yv_pred = np.argmax(model.predict(Xva, batch_size=256, verbose=0), axis=1)
    yt_pred = np.argmax(model.predict(Xte, batch_size=256, verbose=0), axis=1)

    # Reports (print + save) with F1s
    val_metrics = save_and_show_reports(yva, yv_pred, labels=list(range(n_classes)),
                                        out_txt=run_dir / "val_report.txt")
    test_metrics = save_and_show_reports(yte, yt_pred, labels=list(range(n_classes))),
    # fix small tuple issue
    test_metrics = save_and_show_reports(yte, yt_pred, labels=list(range(n_classes)),
                                         out_txt=run_dir / "test_report.txt")

    # Save final model (.keras + .h5 float32)
    final_keras = run_dir / "final.keras"
    model.save(final_keras, include_optimizer=False)

    # Ensure float32 weights for Cube.AI
    model_f32 = keras.models.clone_model(model)
    model_f32.build(model.input_shape)
    model_f32.set_weights([w.astype(np.float32) for w in model.get_weights()])
    h5_path = run_dir / "final_export_cubeai.h5"
    model_f32.save(h5_path, include_optimizer=False)
    print("[SAVED]", final_keras)
    print("[SAVED]", h5_path)

    # Save training log (include F1s)
    with open(run_dir / "train_summary.json", 'w') as f:
        json.dump({
            "preset": preset,
            "L": L,
            "channels": int(n_channels),
            "val": {
                "loss": float(va_eval[0]),
                "acc": float(va_eval[1]),
                "f1_macro": val_metrics["f1_macro"],
                "f1_weighted": val_metrics["f1_weighted"]
            },
            "test": {
                "loss": float(te_eval[0]),
                "acc": float(te_eval[1]),
                "f1_macro": test_metrics["f1_macro"],
                "f1_weighted": test_metrics["f1_weighted"]
            },
            "class_weight": cw,
            "history": {k: [float(x) for x in v] for k, v in hist.history.items()}
        }, f, indent=2)


if __name__ == "__main__":
    for preset in PRESETS:
        try:
            train_one_preset(NPZ_NAME(preset), preset)
        except Exception as e:
            print(f"[ERROR] {preset}: {e}")



========== Training 6ch ==========
Epoch 1/50

Epoch 1: val_accuracy improved from -inf to 0.39610, saving model to /content/tcn_runs/6ch_L128_6ch/best.keras
20/20 - 22s - 1s/step - accuracy: 0.5620 - loss: 0.6961 - val_accuracy: 0.3961 - val_loss: 0.9228 - learning_rate: 0.0030
Epoch 2/50

Epoch 2: val_accuracy did not improve from 0.39610
20/20 - 0s - 15ms/step - accuracy: 0.6034 - loss: 0.6111 - val_accuracy: 0.2208 - val_loss: 1.5804 - learning_rate: 0.0030
Epoch 3/50

Epoch 3: val_accuracy improved from 0.39610 to 0.67532, saving model to /content/tcn_runs/6ch_L128_6ch/best.keras
20/20 - 0s - 20ms/step - accuracy: 0.6334 - loss: 0.5847 - val_accuracy: 0.6753 - val_loss: 0.7058 - learning_rate: 0.0030
Epoch 4/50

Epoch 4: val_accuracy improved from 0.67532 to 0.69481, saving model to /content/tcn_runs/6ch_L128_6ch/best.keras
20/20 - 0s - 19ms/step - accuracy: 0.6423 - loss: 0.5708 - val_accuracy: 0.6948 - val_loss: 0.6518 - learning_rate: 0.0030
Epoch 5/50

Epoch 5: val_accuracy d